# Stage 4 — End-to-end 파이프라인 (PDF → view → element → Donut → JSON)

도면 PDF 1장을 넣으면 전체 파이프라인을 통과시켜 구조화 JSON 을 조립합니다.

```
PDF ──rasterize──▶ page.png
   ──view YOLO(AABB)──▶ view 크롭들
   ──element YOLO-OBB──▶ rectify 된 element 크롭들 (+ type)
   ──element Donut──▶ 값
   ──assemble──▶ { "views": [ { "elements": [ {type, value, box} ] } ] }
```

**커널**: 전 과정을 **단일 커널 `kardi_env`** 로 실행합니다. `kardi_env` 에 YOLO(ultralytics)와
Donut(transformers/torch) 의존성이 모두 설치돼 있어 **커널 전환이 필요 없습니다**.
- **Part A**: 검출·크롭 → `data/_pipeline_tmp/` 에 크롭 + `meta.json` 저장
- **Part B**: 크롭을 Donut 으로 읽어 값 채우고 최종 JSON 조립

→ Part A → Part B 를 같은 커널에서 위→아래로 이어서 실행하면 됩니다.
(`meta.json`/크롭은 디버깅용 중간 산출물일 뿐 커널 간 핸드오프 목적이 아닙니다.)

## Part A — 검출·크롭 (커널: **kardi_env**)

In [ ]:
# ── 설정 ─────────────────────────────────────────────────────────
from pathlib import Path
import json, sys
sys.path.append("detection")                     # crop_utils
from crop_utils import save_crops_from_result
from ultralytics import YOLO

PDF_PATH   = Path("../data/raw_pdf/A136641282.pdf")  # ← held-out 도면으로 바꾸세요
TMP        = Path("../data/_pipeline_tmp"); TMP.mkdir(parents=True, exist_ok=True)
VIEW_PT    = "detection/view/runs/view/weights/best.pt"
ELEM_PT    = "detection/element/runs/element/weights/best.pt"
DPI        = 300

In [ ]:
# ── 1) PDF → page.png ────────────────────────────────────────────
import fitz
doc = fitz.open(PDF_PATH)
pix = doc[0].get_pixmap(matrix=fitz.Matrix(DPI/72, DPI/72), alpha=False)
page_png = TMP / f"{PDF_PATH.stem}.png"
pix.save(page_png); doc.close()
print("page:", page_png)

In [ ]:
# ── 2) view 검출 → view 크롭 ─────────────────────────────────────
view_model = YOLO(VIEW_PT)
vr = view_model.predict(page_png, imgsz=1280, conf=0.25, verbose=False)[0]
view_meta = save_crops_from_result(vr, TMP / "views", PDF_PATH.stem, pad=4)
print(f"view {len(view_meta)}개")

In [ ]:
# ── 3) 각 view 크롭 → element 검출 → rectify 크롭 ────────────────
elem_model = YOLO(ELEM_PT)
records = []   # 최종 조립용 메타
for vi, vm in enumerate(view_meta):
    er = elem_model.predict(vm["path"], imgsz=1024, conf=0.25, verbose=False)[0]
    elems = save_crops_from_result(er, TMP / "elements" / f"view{vi}", f"v{vi}", pad=2)
    records.append({"view_index": vi, "view": vm, "elements": elems})

meta_path = TMP / "meta.json"
json.dump(records, open(meta_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
n_elem = sum(len(r["elements"]) for r in records)
print(f"element 총 {n_elem}개 → {meta_path}")

## Part B — Donut 값 추론 + JSON 조립 (커널: **kardi_env**)

In [ ]:
# ── Donut 로드 + 추론 함수 ───────────────────────────────────────
import json, sys, torch
from pathlib import Path
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel
sys.path.append("detection")                     # 공용 토큰 I/O (학습 노트북과 동일 로직)
from donut_utils import decode_donut_output

TMP        = Path("../data/_pipeline_tmp")
CKPT       = "../checkpoints_elements/final"
TASK       = "<s_element>"
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
proc       = DonutProcessor.from_pretrained(CKPT)
emodel     = VisionEncoderDecoderModel.from_pretrained(CKPT).to(device).eval()

@torch.inference_mode()
def read_value(img_path):
    image = Image.open(img_path).convert("RGB")
    pv  = proc(image, return_tensors="pt").pixel_values.to(device)
    dec = proc.tokenizer(TASK, add_special_tokens=False, return_tensors="pt").input_ids.to(device)
    out = emodel.generate(pv, decoder_input_ids=dec, max_new_tokens=128,
                          pad_token_id=proc.tokenizer.pad_token_id,
                          eos_token_id=proc.tokenizer.eos_token_id, use_cache=True)
    seq = proc.batch_decode(out, skip_special_tokens=False)[0]
    # 특수 토큰(eos/pad/bos)+task 제거 후 dict 파싱 (공용 헬퍼 — strip 집합을 한 곳에서 관리)
    return decode_donut_output(seq, proc.tokenizer, TASK)

In [ ]:
# ── 조립: meta.json 의 element 크롭마다 값 추론 → 최종 JSON ──────
records = json.load(open(TMP / "meta.json", encoding="utf-8"))
assembled = {"source": str(TMP), "views": []}
for r in records:
    view_out = {"view_index": r["view_index"], "box": r["view"]["xy"], "elements": []}
    for e in r["elements"]:
        parsed = read_value(e["path"])        # {"<type>": "<value>"} 기대
        # YOLO 가 분류한 type 으로 보정: 그 키를 우선, 비어 있으면 첫 값, dict 가 아니면 그대로
        if isinstance(parsed, dict):
            value = parsed.get(e["name"]) or next(iter(parsed.values()), parsed)
        else:
            value = parsed
        view_out["elements"].append({"type": e["name"], "value": value,
                                      "conf": round(e["conf"], 3), "box": e["xy"]})
    assembled["views"].append(view_out)

print(json.dumps(assembled, ensure_ascii=False, indent=2)[:2000])
out_path = TMP / "result.json"
json.dump(assembled, open(out_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\n저장:", out_path)